In [1]:
import cv2
import mediapipe as mp
import numpy as np
import time
import joblib
from math import sqrt

In [2]:
model = joblib.load("driver_model.pkl")

In [3]:
mp_face_mesh = mp.solutions.face_mesh
mp_hands = mp.solutions.hands

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    refine_landmarks=True,
    max_num_faces=1
)

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1
)


I0000 00:00:1769083785.373748       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
I0000 00:00:1769083785.379420       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2


In [4]:
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles


In [5]:
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

LEFT_IRIS = [468, 469, 470, 471]
RIGHT_IRIS = [472, 473, 474, 475]

NOSE_TIP = 1


In [6]:
def euclidean(p1, p2):
    return sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def eye_aspect_ratio(eye):
    A = euclidean(eye[1], eye[5])
    B = euclidean(eye[2], eye[4])
    C = euclidean(eye[0], eye[3])
    return (A + B) / (2.0 * C)


In [7]:
def get_head_pose(image, face):
    h, w, _ = image.shape

    image_points = np.array([
        (face[1].x * w, face[1].y * h),
        (face[152].x * w, face[152].y * h),
        (face[33].x * w, face[33].y * h),
        (face[263].x * w, face[263].y * h),
        (face[61].x * w, face[61].y * h),
        (face[291].x * w, face[291].y * h)
    ], dtype="double")

    model_points = np.array([
        (0, 0, 0),
        (0, -63.6, -12.5),
        (-43.3, 32.7, -26.0),
        (43.3, 32.7, -26.0),
        (-28.9, -28.9, -24.1),
        (28.9, -28.9, -24.1)
    ])

    focal_length = w
    center = (w / 2, h / 2)
    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ])

    _, rvec, _ = cv2.solvePnP(
        model_points, image_points, camera_matrix, np.zeros((4, 1))
    )

    rmat, _ = cv2.Rodrigues(rvec)
    angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)

    return angles[0]*360, angles[1]*360, angles[2]*360


In [8]:
PHONE_DIST_THRESH = 70
PHONE_ANGLE_THRESH = 40

HEAD_PITCH_DOWN_THRESH = -12   # head down
GAZE_Y_DOWN_THRESH = 6         # eyes down


In [ ]:
cap = cv2.VideoCapture(0)
time.sleep(1)

if not cap.isOpened():
    raise RuntimeError("Camera not opened")

print("Driver Monitoring Started (Press Q to quit)")

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    display_frame = frame.copy()
    h, w, _ = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(rgb)
    hand_results = hands.process(rgb)

    status = "NO FACE"
    color = (0, 255, 255)
    phone_usage = False
    rule_distracted = False

    if face_results.multi_face_landmarks:
        face = face_results.multi_face_landmarks[0].landmark

        def get_point(i):
            return np.array([face[i].x * w, face[i].y * h])

        # 🟢 Draw FULL FACE LANDMARKS
        for lm in face:
            cv2.circle(
                display_frame,
                (int(lm.x * w), int(lm.y * h)),
                1,
                (0, 255, 0),
                -1
            )

        # 🟢 Draw IRIS LANDMARKS
        for idx in LEFT_IRIS + RIGHT_IRIS:
            cv2.circle(
                display_frame,
                (int(face[idx].x * w), int(face[idx].y * h)),
                3,
                (0, 255, 0),
                -1
            )

        # Face bounding box
        xs = [lm.x for lm in face]
        ys = [lm.y for lm in face]
        x_min, x_max = int(min(xs)*w), int(max(xs)*w)
        y_min, y_max = int(min(ys)*h), int(max(ys)*h)

        # EAR
        left_eye = [get_point(i) for i in LEFT_EYE]
        right_eye = [get_point(i) for i in RIGHT_EYE]
        ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

        eye_state = 0 if ear < 0.18 else 1 if ear < 0.23 else 2

        # Gaze
        try:
            gaze_y = (
                np.mean([get_point(i)[1] for i in LEFT_IRIS + RIGHT_IRIS]) -
                np.mean([get_point(i)[1] for i in LEFT_EYE + RIGHT_EYE])
            )
        except:
            gaze_y = 0

        # Head pose
        pitch, yaw, roll = get_head_pose(frame, face)

        # Rule-based unsafe
        if pitch < HEAD_PITCH_DOWN_THRESH or gaze_y > GAZE_Y_DOWN_THRESH:
            rule_distracted = True

        # Hand / phone
        hand_face_dist = 0
        if hand_results.multi_hand_landmarks:

            # 🟢 DRAW HAND LANDMARK POINTS
            for hand_lms in hand_results.multi_hand_landmarks:
                for lm in hand_lms.landmark:
                    cv2.circle(
                        display_frame,
                        (int(lm.x * w), int(lm.y * h)),
                        4,                 # bigger dots
                        (0, 255, 0),
                        -1
                    )

            hand = hand_results.multi_hand_landmarks[0].landmark
            hand_face_dist = euclidean(
                np.array([hand[8].x*w, hand[8].y*h]),
                get_point(NOSE_TIP)
            )

        # ML prediction
        features = np.array([[ear, eye_state, 0, gaze_y, pitch, yaw, roll, hand_face_dist]])
        prediction = model.predict(features)[0]

        if prediction == 1 or rule_distracted:
            status = "DISTRACTED"
            color = (0, 0, 255)
        else:
            status = "SAFE"
            color = (0, 255, 0)

        # 🔴 EXTRA BOLD FACE BOX
        cv2.rectangle(
            display_frame,
            (x_min, y_min),
            (x_max, y_max),
            color,
            6   # thicker box
        )

        # 🔴 EXTRA BOLD LABEL
        cv2.putText(
            display_frame,
            status,
            (x_min, y_min - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.5,   # bigger text
            color,
            4      # thicker text
        )

    cv2.imshow("Driver Monitoring", display_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Driver Monitoring Started (Press Q to quit)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.